# ITCC508 Lab Exercise PT-M1
## Building and Evaluating a Domain-Specific RAG Chatbot

**Course:** ITC CS08 — Introduction to LLMOps and RAG Concepts
**Topic:** Retrieval-Augmented Generation (RAG) & Hallucination Mitigation
**Domain:** Republic Act No. 9003 — Ecological Solid Waste Management Act of 2000
**Stack:** LangChain · HuggingFace `all-MiniLM-L6-v2` · ChromaDB · Groq

**Name: Audrey Benjamin Gutierrez**
**Section: 401 I**
**Date: 9/9/2026**

---
Every code cell has a markdown cell above it explaining what the code does and why it is there, as the lab requires.

Beyond the required cells, this notebook contains three additions, each clearly labelled where it appears:

1. **Cell 7b** — three pointed in-domain queries with verifiable answers, added because the template's generic "summarize the documents" prompt turned out to be a weak test of retrieval.
2. **A retrieval-failure investigation** — one of those three queries was refused even though the answer exists in the corpus. Rather than ignore it, I traced the cause down to the specific chunk boundary responsible.
3. **Config C** — a third prompt configuration beyond the two the lab specifies, added because the required Task 3 comparison changes two variables at once and cannot show which one matters.

### Two deviations from the supplied template

Both were forced by the environment rather than chosen, and both are documented at the cell where they occur:

- **The model was substituted.** `llama-3.1-8b-instant` was decommissioned by Groq on 16 August 2026. All results below were produced on `openai/gpt-oss-20b`, the replacement Groq recommends.
- **Two import paths changed.** LangChain 1.0 moved `langchain.chains` into a separate `langchain-classic` package. The function calls are unchanged; only the module path differs.

### Running order

Cells are numbered in execution order. Cell 3 requires the seven source files to already be present in `my_data/` — it creates the folder but does not populate it. Note also that Colab clears uploaded files when the runtime disconnects, so the files need re-uploading after any restart.

---
## Cell 1 — Environment Setup & Package Installation

This cell installs every third-party library the pipeline depends on. It is a shell command, not Python: the leading `!` tells Jupyter/Colab to hand the line to the system shell.

Line-by-line:

- `!pip install -q ...` — `pip` is Python's package installer. The `-q` (quiet) flag suppresses the long progress log so the notebook output stays readable.
- `langchain` — the core framework. Supplies the chain abstractions (`create_retrieval_chain`) that wire retrieval and generation together.
- `langchain-community` — community-maintained integrations. This is where `DirectoryLoader` and the `Chroma` vector-store wrapper live.
- `langchain-groq` — the provider adapter that lets LangChain talk to Groq's inference API through the `ChatGroq` class.
- `langchain-huggingface` — the adapter exposing HuggingFace embedding models to LangChain via `HuggingFaceEmbeddings`.
- `chromadb` — the vector database engine itself. It stores the embedding vectors and performs the similarity search.
- `pypdf` — pure-Python PDF parser. Needed so the loader can extract text from `.pdf` source files.
- `unstructured` — general-purpose document parser. `DirectoryLoader` falls back to it for mixed file types, which is what the `**/*.*` glob in Cell 3 asks for.

**Why it is necessary:** a RAG pipeline is an assembly of four separable concerns — document parsing, embedding, vector storage, and generation. Each package supplies exactly one of those. Nothing downstream imports successfully until this cell completes.

In [ ]:
# Cell 1: Environment Setup & Package Installation
!pip install -q langchain langchain-community langchain-groq langchain-huggingface chromadb pypdf unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 20.8 MB/s eta 0:00:00
 

---
## Cell 2 — Secure API Key Configuration

This cell loads the Groq credential into the process environment without ever writing it into a saved cell.

Line-by-line:

- `import os` — gives access to `os.environ`, the dictionary of environment variables for the running Python process. LangChain's `ChatGroq` reads `GROQ_API_KEY` from here automatically, so no key ever has to be passed as a function argument.
- `import getpass` — supplies `getpass.getpass()`, an input function that masks what the user types and, critically, does **not** echo the value into the notebook's stored output.
- `if "GROQ_API_KEY" not in os.environ:` — guards against re-prompting. If the key is already set (because the cell was run earlier in the same session), the prompt is skipped.
- `os.environ["GROQ_API_KEY"] = getpass.getpass(...)` — reads the key from the masked prompt and stores it in the environment for the lifetime of this runtime only. When the runtime is recycled, the key is gone.

**Why it is necessary:** the alternative — `api_key = "gsk_..."` written in a cell — persists the secret inside the `.ipynb` file. Since the notebook is a submitted deliverable and is often pushed to GitHub, that hardcoded key would leak with it. This pattern is the minimum viable secret-handling practice and is one of the points to argue in Section IV of the report.

In [ ]:
# Cell 2: Secure API Key Configuration
import os
import getpass

# Prompts for API key securely without saving or echoing plain text in notebook cells
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


---
## Cell 3 — Loading Custom Data & Chunking

This cell turns raw files on disk into a list of small, overlapping text chunks — the units that will later be embedded and retrieved.

Line-by-line:

- `from langchain_community.document_loaders import DirectoryLoader` — imports the loader that walks a folder and parses every matching file into LangChain `Document` objects. Each `Document` has `.page_content` (the text) and `.metadata` (source path, page number, etc.).
- `from langchain_text_splitters import RecursiveCharacterTextSplitter` — imports the chunking strategy described below.
- `os.makedirs("my_data", exist_ok=True)` — creates the source folder. `exist_ok=True` makes the call idempotent, so re-running the cell does not raise `FileExistsError`. **This creates an empty folder only — upload your documents into it before running.**
- `loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)` — targets the folder. `glob="**/*.*"` matches any file with an extension at any depth (`**` recurses into subfolders). `show_progress=True` prints a progress bar, which is useful because parsing is the slowest step here.
- `raw_documents = loader.load()` — executes the parsing and returns a list of `Document` objects, typically one per page for PDFs and one per file for `.txt`.
- `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` — configures the splitter. `chunk_size=500` caps each chunk at roughly 500 characters. `chunk_overlap=50` repeats the last 50 characters of each chunk at the start of the next one.
- `documents = text_splitter.split_documents(raw_documents)` — applies the splitter, carrying the original `.metadata` down onto every derived chunk so source attribution survives chunking.
- `print(f"Loaded {len(raw_documents)} ...")` — a sanity check. If this prints `0 raw document(s)`, the `my_data/` folder is empty and every later cell will fail.

**Why chunking is necessary:** embedding models have a fixed input window and produce a single vector per input. Embedding a whole 30-page PDF yields one blurred vector that matches nothing precisely. Small chunks give sharp, topically-focused vectors, so similarity search can return the specific passage that answers the question rather than the whole document.

**Why the splitter is "recursive":** it tries a prioritised list of separators — paragraph breaks, then line breaks, then sentences, then words — and only falls back to a harder cut when a chunk still exceeds the size limit. That keeps semantically related sentences together instead of slicing mid-thought.

**Why overlap is necessary:** a hard boundary can land in the middle of a definition, leaving the subject in chunk *n* and the predicate in chunk *n+1*, so neither chunk answers the question alone. The 50-character overlap is a cheap insurance margin against that.

> **Debugging note:** with `glob="**/*.*"`, `DirectoryLoader` defaults to the `unstructured` parser. For plain `.txt` files this works but is slower than necessary, and on some versions `unstructured` triggers an `nltk` data download on first use, which shows up as a delay or an `LookupError`. If Cell 3 hangs or errors, the standard fix is to constrain the glob and pass an explicit loader class:
>
> ```python
> from langchain_community.document_loaders import TextLoader
> loader = DirectoryLoader("my_data/", glob="**/*.txt",
>                          loader_cls=TextLoader,
>                          loader_kwargs={"encoding": "utf-8"},
>                          show_progress=True)
> ```
>
> `TextLoader` reads the file directly with no parsing layer, so it cannot fail the way `unstructured` can. (The PDF equivalent is `PyPDFLoader` with `glob="**/*.pdf"`.) Record the fix in the report's references if you needed it.

In [ ]:
# Cell 3: Loading Custom Data & Chunking
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create target data directory
os.makedirs("my_data", exist_ok=True)

# Load all documents from the directory
loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)
raw_documents = loader.load()

# Split documents into smaller semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = text_splitter.split_documents(raw_documents)
print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")

100%|██████████| 7/7 [00:18<00:00,  2.59s/it]

Loaded 7 raw document(s) and split into 239 chunks.


---
## Cell 4 — Embedding Model & Vector DB Indexing

This cell converts every text chunk into a numeric vector, stores those vectors in a searchable index, and exposes the index as a retriever.

Line-by-line:

- `from langchain_huggingface import HuggingFaceEmbeddings` — imports the wrapper that downloads a sentence-transformer model and runs it locally.
- `from langchain_community.vectorstores import Chroma` — imports the LangChain interface to the ChromaDB vector database.
- `embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` — loads the embedding model. `all-MiniLM-L6-v2` is a 6-layer distilled sentence transformer that maps text to a **384-dimensional** vector. The first run downloads roughly 90 MB of weights; later runs use the cache. It runs on CPU, needs no API key, and costs nothing per call — which is why the exercise specifies it over a hosted embedding API.
- `vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)` — a convenience constructor that does three things in one call: embeds every chunk from Cell 3, builds the index, and returns the populated store. With no `persist_directory` argument, the database lives **in memory only** and is discarded when the runtime restarts, so this cell must be re-run after every restart.
- `retriever = vectorstore.as_retriever(search_kwargs={"k": 3})` — wraps the store in LangChain's standard `Retriever` interface. `k: 3` fixes the top-k: every query returns the 3 most similar chunks.

**Why embeddings are necessary:** keyword search fails when the user's wording differs from the document's wording. An embedding model places semantically similar text near each other in vector space, so "how much does enrolment cost" can retrieve a passage that says "tuition fees are…" despite sharing almost no words. Similarity is measured as distance between vectors.

**Why the top-k value matters:** `k` is the central precision/recall dial of the retriever. Too low and the answer-bearing chunk may be missed, forcing the fallback even for valid in-domain questions. Too high and unrelated chunks pad the context window, diluting the signal and giving the model more irrelevant material to drift toward. `k=3` is a reasonable default for a 500-character chunk size. **Record `k=3` in the Implementation Details table (Report Section II).**

In [ ]:
# Cell 4: Embedding Model & Vector DB Indexing
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize open-source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store embeddings into Chroma vector database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

# Set vectorstore as a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

---
## Cell 5 — Model Initialization and Domain System Prompt

This cell configures the generator half of RAG: which model answers, how randomly it answers, and under what rules.

Line-by-line:

- `from langchain_groq import ChatGroq` — imports the chat-model class for Groq's inference API.
- `from langchain_core.prompts import ChatPromptTemplate` — imports the template class that assembles a structured, multi-role message list.
- `llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)` — instantiates the small language model. The API key is picked up automatically from the `GROQ_API_KEY` environment variable set in Cell 2. `temperature=0` makes decoding effectively greedy: at each step the highest-probability token is chosen, so the same input yields the same output.
- `system_prompt = (...)` — a single string built by implicit concatenation of adjacent string literals. It carries three separate instructions: assign the model a narrow role; **restrict** it to the supplied context; and specify an exact **fallback sentence** for unanswerable questions.
- `"Context:\n{context}"` — the placeholder that LangChain fills at run time with the text of the retrieved chunks. Without this placeholder the retrieved context never reaches the model and the system degrades into a plain, ungrounded chatbot.
- `prompt = ChatPromptTemplate.from_messages([...])` — builds the final message list: the `"system"` turn carries the rules and the context, the `"human"` turn carries `{input}`, which is filled with the user's question.

**Why `temperature=0` is necessary here:** temperature rescales the probability distribution before sampling. At 0 the model always takes the most likely continuation; as it rises, lower-probability tokens become reachable. For a factual, document-grounded assistant that is pure downside — the "creativity" it buys is exactly the freedom to depart from the source text. It is also a requirement for a fair experiment: a deterministic baseline means any behaviour change in Task 3 is attributable to the configuration change, not to sampling noise.

**Why the role/restriction/fallback triple is necessary:** the model's pretrained knowledge does not disappear when you hand it context. Absent an explicit instruction, it will happily answer from parametric memory and present that answer with the same confidence as a grounded one. The restriction narrows the permitted source to the context; the fallback gives the model a legitimate, low-effort exit when the context is insufficient. That exit matters — without a sanctioned way to say "no", producing *something* is the only path the model has.

In [ ]:
# Cell 5: Model Initialization and Domain System Prompt
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialize the SLM
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0
)

# Custom domain system prompt
system_prompt = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n"
    "If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

---
## Cell 6 — Pipeline Assembly

This cell joins the retriever from Cell 4 and the prompt/LLM from Cell 5 into one callable object.

Line-by-line:

- `from langchain.chains import create_retrieval_chain` — imports the outer chain builder.
- `from langchain.chains.combine_documents import create_stuff_documents_chain` — imports the builder for the document-combining step.
- `combine_docs_chain = create_stuff_documents_chain(llm, prompt)` — builds the generation half. "Stuff" names the combination strategy: take the retrieved `Document` objects, concatenate their `page_content`, and **stuff** the result into the `{context}` placeholder in one single call to the model. The alternatives (map-reduce, refine) make several calls and exist for document sets too large for one context window; with `k=3` and 500-character chunks, stuffing is both cheaper and lossless.
- `rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` — builds the full pipeline and defines the data flow: take the `input` question → pass it to the retriever → receive the top-3 chunks → hand question and chunks to `combine_docs_chain` → return a dictionary containing `input`, `context`, and `answer`.

**Why this assembly is necessary:** it is the "augmented" in Retrieval-Augmented Generation. Retrieval alone returns passages a user must read themselves; generation alone answers fluently but ungroundedly. Only the composition produces a natural-language answer traceable to specific source chunks. Note also that this chain **captures** the `retriever` and `prompt` objects that existed when this cell ran — which is precisely why Task 3 requires re-running this cell after modifying Cell 5.

---
### Debugging note — `ModuleNotFoundError: No module named 'langchain.chains'`

Running Cell 6 as supplied raised:

```
ModuleNotFoundError: No module named 'langchain.chains'
```

The cause is a breaking change in the framework, not an error in the lab code. LangChain 1.0 restructured the project into modular packages and moved the legacy chain builders — including `create_retrieval_chain` and `create_stuff_documents_chain` — out of the main `langchain` package and into `langchain-classic`. The lab template was written against LangChain 0.x, and `pip` installed 1.x.

The fix has two parts: install the package below, then change the two import paths in Cell 6 from `langchain.chains` to `langchain_classic.chains`. Nothing else changes — the functions themselves are identical, and `llm`, `prompt` and `retriever` are all untouched.

This is worth recording as an LLMOps observation in its own right. A working pipeline stopped working because an upstream dependency reorganised its namespace between major versions. Pinning versions in the install cell (`langchain==0.3.x`) would have avoided it; so would migrating, which is what I did here.

In [ ]:
!pip install -q langchain-classic

### Cell 6 — Pipeline Assembly (with the migrated import paths)

The two import lines below are the only change from the supplied template: `langchain.chains` becomes `langchain_classic.chains`, for the reason given above. `create_stuff_documents_chain(llm, prompt)` and `create_retrieval_chain(retriever, combine_docs_chain)` are called exactly as the template specifies, with the same three objects built in Cells 4 and 5.

Worth restating here because it matters twice later: `rag_chain` **captures** the `llm`, `prompt` and `retriever` objects as they exist at the moment this cell runs. It does not hold live references that update when those variables are reassigned. That is why the model substitution required re-running this cell, and why Task 3 requires re-running it again after modifying the prompt — editing Cell 5 alone would change nothing observable.

In [ ]:
# Cell 6: Pipeline Assembly
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Combine prompt and LLM to process context
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Assemble full retrieval-augmented generation chain
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

---
## Cell 7 — Testing Your Custom Domain Chatbot (Test Case 1: In-Domain Query)

This cell runs the assembled chain against a question the documents *should* be able to answer, and prints both the answer and its provenance.

Line-by-line:

- `user_query = "Summarize the key information found in the documents."` — the in-domain test question. **Replace this with a specific question about your own dataset.** A generic "summarize everything" is a weak test: with `k=3` the retriever only ever sees three chunks, so a broad query retrieves an arbitrary three and the summary looks thin. A pointed factual question ("What are the requirements for X?") exercises retrieval far better and gives you a checkable expected answer for Table I.
- `response = rag_chain.invoke({"input": user_query})` — executes the pipeline. The input dictionary key must be `"input"` because that is the placeholder name declared in the human message in Cell 5. The return value is a dictionary.
- `print(response["answer"])` — the generated answer string.
- `for i, doc in enumerate(response["context"]):` — iterates the `Document` objects the retriever actually selected. `enumerate` supplies the counter `i`.
- `doc.metadata.get("source", "Unknown")` — reads the source filename from the metadata that has been carried along since Cell 3. `.get()` with a default is used instead of `[...]` so a chunk with missing metadata prints `Unknown` rather than raising `KeyError`.

**Why inspecting the retrieved chunks is necessary:** it separates two very different failure modes that look identical from the answer alone. If the answer is wrong and the correct chunk *was* retrieved, the generation step is at fault — a prompt or model problem. If the correct chunk was never retrieved, the fault lies upstream in chunking, embedding, or `k` — and no amount of prompt engineering will fix it. Printing sources also gives the reader a way to verify the answer, which is the practical anti-hallucination benefit of RAG over a bare LLM.

In [ ]:
# Cell 7: Testing Your Custom Domain Chatbot
# Test Case 1: In-Domain Query
user_query = "Summarize the key information found in the documents."
response = rag_chain.invoke({"input": user_query})

print("--- DOMAIN QUERY ANSWER ---")
print(response["answer"])
print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(response["context"]):
    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
**Key Points from the Documents**

1. **Record‑Keeping Requirements**  
   - Maintain records of personnel training.  
   - Keep a copy of written notifications to the Department, local health agency, and fire authority that list the operator or responsible party’s names, addresses, and telephone numbers.  
   - Document all approvals, determinations, and other requirements issued by the Department.  
   - Preserve records of water‑quality monitoring for surface and ground waters, effluent, and gas emissions.  
   - Retain any required signs and related documentation (Sections 40 and 41 of the Act).

2. **Public Education & Information**  
   - The Local Government Unit (LGU) must develop an education and public‑information component.  
   - This component should explain how the LGU will educate citizens on source‑reduction, recycling, and composting programs.  
   - The plan must ensure that information on waste‑collection services, solid‑waste management, 

---
## Cell 7b — Additional in-domain test queries

Cell 7 above uses the template's generic "summarize the documents" query. That is a weak test of retrieval: with `k=3` the retriever only ever sees three chunks out of roughly 240, so a broad summarisation request retrieves an essentially arbitrary three and the answer looks thin through no fault of the pipeline.

The cell below asks three pointed questions instead, each with a verifiable answer in the source text. These are the rows that belong in Table I of the report, because for each one you can state an expected answer in advance and check the model against it:

| Query | Expected answer | Source chapter |
|-------|-----------------|----------------|
| Penalty for littering in a public place | Fine of ₱300–₱1,000, or community service of 1–15 days, or both | `CHAPTER_VI.txt` (Sec. 49(a)) |
| Composition of the National Solid Waste Management Commission | 14 members from government, 3 from the private sector | `CHAPTER_II.txt` (Sec. 4) |
| How the Solid Waste Management Fund is sourced | Fines and penalties, permit/licence proceeds, donations, grants, and GAA appropriations | `CHAPTER_V.txt` (Sec. 46) |

Note the second and third questions target different chapters. If the printed source filename matches the expected chapter each time, that is direct evidence your retrieval is working — not just that the model produced a plausible answer.

In [ ]:
# Cell 7b: Additional in-domain test queries (pointed, verifiable)
in_domain_queries = [
    "What is the penalty for littering or dumping waste in a public place?",
    "Who are the members of the National Solid Waste Management Commission?",
    "Where does the Solid Waste Management Fund get its money from?",
]

for q in in_domain_queries:
    r = rag_chain.invoke({"input": q})
    print("=" * 70)
    print("QUERY:", q)
    print("-" * 70)
    print(r["answer"])
    print("\nRetrieved from:", [d.metadata.get("source", "Unknown") for d in r["context"]])
    print()

QUERY: What is the penalty for littering or dumping waste in a public place?
----------------------------------------------------------------------
I cannot answer based on the provided domain data.

Retrieved from: ['my_data/CHAPTER VI.txt', 'my_data/chapter 1.txt', 'my_data/CHAPTER III.txt']

QUERY: Who are the members of the National Solid Waste Management Commission?
----------------------------------------------------------------------
The National Solid Waste Management Commission is composed of:

- **Fourteen (14) members from the government sector** – these are the heads of the relevant government agencies, serving in their ex‑officio capacity.  
- **Three (3) members from the private sector** – appointed on the basis of integrity, professionalism, and distinguished service in environmental and related fields.

The **Department Secretary** serves as the **chairman** of the Commission, and a **private‑sector representative** serves as the **vice‑chairman**.

Retrieved from: ['my

---
---
# Retrieval Failure Investigation
### (additional analysis, not required by the lab)
---

Cell 7b returned two correct answers and one refusal. The refusal is the interesting one.

Asked *"What is the penalty for littering or dumping waste in a public place?"*, the system replied `I cannot answer based on the provided domain data.` — despite the fact that RA 9003 answers this question directly in Section 49(a): a fine of ₱300 to ₱1,000, or community service of one to fifteen days.

This is a **false refusal**: the model declined a question the corpus can actually answer. It is the mirror image of hallucination, and it is a real cost of strict grounding. Since `CHAPTER VI.txt` *was* among the retrieved sources, something more specific than "retrieval missed the chapter" is going on.

The three cells that follow trace the cause. The first prints the full text of what the retriever actually handed to the model.

In [ ]:
# Diagnostic: what was actually retrieved for the littering query?
q = "What is the penalty for littering or dumping waste in a public place?"
for i, d in enumerate(retriever.invoke(q)):
    print(f"--- CHUNK {i+1} | {d.metadata.get('source')} ---")
    print(d.page_content)
    print()

--- CHUNK 1 | my_data/CHAPTER VI.txt ---
CHAPTER VI

Penal Provisions

Section 48. Prohibited Acts. – The following acts are prohibited:

(1) Littering, throwing, dumping of waste matters in public places, such as roads,

sidewalks, canals, esteros or parks, and establishment, or causing or

permitting the same;

(2) Undertaking activities or operating, collecting or transporting equipment in

violation of sanitation operation and other requirements or permits set forth in

or established pursuant to this Act;

--- CHUNK 2 | my_data/chapter 1.txt ---
(s) Municipal waste shall refer to wastes produced from activities within local

government units which include a combination of domestic, commercial,

institutional and industrial wastes and street litters;

(t) Open dump shall refer to a disposal area wherein the solid wastes are

indiscriminately thrown or disposed of without due planning and

consideration for environmental and health standards;

(u) Opportunity to recycle shall refer 

**What the first diagnostic shows.** Chunk 1 comes from `CHAPTER VI.txt` and contains the chapter heading plus Section 48 items (1) and (2) — it stops mid-list, well before Section 49 where the peso amounts appear. Chunk 2 is a definitions passage from Chapter I that matched on the word *litters*. Chunk 3 is unrelated Chapter III text.

So the model was asked for a penalty and given three chunks containing no penalty. Refusing was the correct behaviour; the fault is upstream in retrieval.

The obvious first hypothesis is that `k=3` is simply too small a window, and that retrieving more neighbours would eventually reach Section 49. The next cell tests that by doubling `k` to 6, holding everything else constant.

In [ ]:
# Does a larger k recover the penalty provision?
retriever_k6 = vectorstore.as_retriever(search_kwargs={"k": 6})
chain_k6 = create_retrieval_chain(retriever_k6, combine_docs_chain)

r6 = chain_k6.invoke({"input": q})
print("--- ANSWER WITH k=6 ---")
print(r6["answer"])
print("\nRetrieved from:", [d.metadata.get("source") for d in r6["context"]])

--- ANSWER WITH k=6 ---
I cannot answer based on the provided domain data.

Retrieved from: ['my_data/CHAPTER VI.txt', 'my_data/chapter 1.txt', 'my_data/CHAPTER III.txt', 'my_data/CHAPTER VI.txt', 'my_data/CHAPTER III.txt', 'my_data/CHAPTER VI.txt']


**Doubling `k` did not fix it.** At `k=6` the system pulled three chunks from `CHAPTER VI.txt` instead of one — it is reaching further into the penal chapter — and still refused.

That rules out simple coverage. If the problem were only that the window was too narrow, a window twice as wide would have caught the answer. The failure must be in *which* chunks are judged similar to the question, not *how many* are returned.

The last diagnostic locates the chunk that actually holds the fine amounts, then tests whether that chunk is reachable at all by querying in the statute's own vocabulary rather than the user's.

In [ ]:
# Where do the peso amounts live, and does any query reach them?
hits = [d for d in documents if "P300.00" in d.page_content or "Three hundred pesos" in d.page_content]
print(f"Chunks containing the fine amounts: {len(hits)}")
for d in hits:
    print("---", d.metadata.get("source"), "---")
    print(d.page_content[:400])
    print()

# Does a query using the statute's own wording retrieve it?
print("=" * 60)
for d in retriever.invoke("fine of not less than Three hundred pesos community service"):
    print(d.metadata.get("source"), "|", d.page_content[:100].replace("\n", " "))

Chunks containing the fine amounts: 1
--- my_data/CHAPTER VI.txt ---
hundred pesos (P300.00) but not more than One thousand pesos (P1,000.00) or render

community service for not less than one (1) day to not more than fifteen (15) days to an

LGU where such prohibited acts are committed, or both;

(b) Any person who violates Sec. 48, pars. (2) and (3), shall, upon conviction, be

punished with a fine of not less than Three hundred pesos (P300.00) but not

more than

my_data/CHAPTER VI.txt | hundred pesos (P300.00) but not more than One thousand pesos (P1,000.00) or render  community servic
my_data/CHAPTER VI.txt | pesos (P100,000.00) but not more than One million pesos (P1,000,000.00),  or imprisonment not less t
my_data/CHAPTER VI.txt | one (1) day to not more than fifteen (15) days, or both;  (b) Any person who violates Sec. 48 pars. 


### What the investigation established

The chunk containing the amounts begins mid-sentence:

> `hundred pesos (P300.00) but not more than One thousand pesos (P1,000.00) or render community service for not less than one (1) day...`

The opening of Section 49(a) — *"Any person who violates Sec. 48, paragraph (1) shall, upon conviction, be punished with a fine of not less than Three..."* — was cut off into the preceding chunk. What remains is a fragment stating amounts with no indication of **which offence they apply to**.

The second half of the cell confirms the chunk is not unreachable: querying in the statute's own wording (*"fine of not less than Three hundred pesos community service"*) retrieves it as the top hit. It simply does not match a natural-language question about littering, because nothing in it mentions littering.

**The full causal chain:**

1. RA 9003 separates the offence (Sec. 48) from its penalty (Sec. 49). They are one legal concept but two statutory sections.
2. `chunk_size=500` split Section 49(a) mid-sentence, orphaning the amounts from their subject.
3. A question about littering therefore retrieves Sec. 48 — which contains the word — but not Sec. 49, which contains the numbers and never repeats the word, referring back only by cross-reference (*"Sec. 48, paragraph (1)"*).
4. Raising `k` did not help, because the problem is semantic, not coverage.
5. Given no penalty in context, the model refused rather than inventing a figure.

**Why this matters beyond this one query.** Embedding-based retrieval matches on surface semantics, and a cross-reference like *"Sec. 48, paragraph (1)"* carries almost no semantic signal. Legal and regulatory text is built out of exactly this kind of internal reference, which makes it unusually hostile to naive fixed-size chunking. The fix belongs at the chunking stage, not the retrieval stage: a larger `chunk_size`, a much larger `chunk_overlap`, or a structure-aware splitter that treats each Section as an atomic unit. Testing those variants is the clearest next step for this system, and I have listed it as future work.

---
---
# 4. Practical Experiments & Deliverables
---

## Task 1: Domain & Dataset Description

**Chosen domain:** Philippine environmental law — Republic Act No. 9003, the *Ecological Solid Waste Management Act of 2000*.

**Problem statement:**

> RA 9003 governs solid waste management throughout the Philippines, and its provisions are consulted routinely by LGU personnel, barangay officials, students and ordinary citizens who need to know what the law requires of them. What they need is specific and consequential: which acts are prohibited, what fine attaches to each, which body holds which mandate, what deadlines bind an LGU. Answering wrongly has real cost.
>
> A general-purpose LLM handles this badly for two reasons. Its knowledge of any single national statute is diffuse — it can describe RA 9003 in general terms but cannot reliably reproduce a specific peso amount or section number. More seriously, it gives no signal when it is guessing: a fabricated fine is delivered in exactly the same confident register as a correct one. For statutory text, where the precise figure *is* the answer, that is unacceptable. A RAG system addresses both halves — it grounds each answer in retrieved text, and it can be instructed to refuse rather than guess.

**Source files uploaded to `./my_data/`:**

| # | Filename | Content |
|---|----------|---------|
| 1 | `chapter 1.txt` | Chapter I — Basic Policies; short title, declaration of policies, definition of terms |
| 2 | `CHAPTER II.txt` | Chapter II — Institutional Mechanism; National Solid Waste Management Commission, National Ecology Center, local SWM boards |
| 3 | `CHAPTER III.txt` | Chapter III — Comprehensive Solid Waste Management; status report, national framework, LGU plans, segregation, MRFs, collection, disposal |
| 4 | `CHAPTER IV.txt` | Chapter IV — Incentives; fiscal and non-fiscal incentives, financial assistance, grants to LGUs |
| 5 | `CHAPTER V.txt` | Chapter V — Financing; Solid Waste Management Fund, authority to collect SWM fees |
| 6 | `CHAPTER VI.txt` | Chapter VI — Penal Provisions; Sec. 48 prohibited acts, Sec. 49 fines and penalties, Sec. 50 administrative sanctions |
| 7 | `CHAPTER VII.txt` | Chapter VII — Miscellaneous Provisions; citizen suits, SLAPP, research, public education, appropriations, effectivity |

The corpus is the complete text of RA 9003, approximately 108,000 characters, obtained as a text-layer PDF and converted to plain text.

**Why split by chapter.** Each file maps to one structural division of the statute, so the source filename printed beside every retrieved chunk immediately identifies which part of the law an answer came from. A single combined file would work identically for retrieval, but every chunk would report the same source and the provenance output in Cell 7 would tell the reader nothing. This choice paid off directly in the investigation section below, where knowing that a chunk came from `CHAPTER VI.txt` was the first step in diagnosing the failure.

**Chunk statistics reported by Cell 3:**

```
Loaded 7 raw document(s) and split into 239 chunks.
```

All seven files parsed, none truncated.

**Format note.** The source files are `.txt` rather than PDF by deliberate choice. A `.txt` file has no parsing layer that can fail, whereas a scanned PDF contains images of words from which `pypdf` extracts nothing. Converting first removed an entire class of loading failure before it could occur.

### Optional verification cell

Run this to confirm the folder contents and inspect one chunk. It is a diagnostic, not part of the required pipeline — it helps you fill in the table above and confirms that metadata survived chunking.

In [ ]:
# Optional: verify dataset contents and inspect one chunk
import os

print("Files in my_data/:")
for f in os.listdir("my_data"):
    print(" -", f, f"({os.path.getsize(os.path.join('my_data', f))} bytes)")

print(f"\nTotal chunks indexed: {len(documents)}")
print("\n--- SAMPLE CHUNK [0] ---")
print("Metadata:", documents[0].metadata)
print("Content:", documents[0].page_content[:300])

Files in my_data/:
 - CHAPTER VI.txt (5293 bytes)
 - CHAPTER II.txt (20822 bytes)
 - CHAPTER IV.txt (4789 bytes)
 - CHAPTER V.txt (2219 bytes)
 - CHAPTER III.txt (48553 bytes)
 - chapter 1.txt (15810 bytes)
 - CHAPTER VII.txt (11002 bytes)

Total chunks indexed: 239

--- SAMPLE CHUNK [0] ---
Metadata: {'source': 'my_data/CHAPTER VI.txt'}
Content: CHAPTER VI

Penal Provisions

Section 48. Prohibited Acts. – The following acts are prohibited:

(1) Littering, throwing, dumping of waste matters in public places, such as roads,

sidewalks, canals, esteros or parks, and establishment, or causing or

permitting the same;

(2) Undertaking activities


---
## Task 2: Out-of-Domain Fallback Test

**Goal:** confirm that the baseline configuration (`temperature=0` **with** the fallback rule) refuses a question your documents cannot answer.

**How to choose a good out-of-domain query:** pick a topic with no lexical or semantic overlap with your dataset at all. If your documents are legal terms of service, ask about planetary science. A near-miss question ("what about clause 9?") is a weaker test, because the retriever will surface genuinely related chunks and the model may answer partially and correctly — which is not a fallback failure.

**Expected output:** `I cannot answer based on the provided domain data.`

Run the cell below, then paste the query and the exact output into the box underneath.

In [ ]:
# Task 2: Out-of-Domain Fallback Test (BASELINE — temperature=0, fallback rule present)
oo_domain_query = "What is the average surface temperature of the planet Venus?"

baseline_response = rag_chain.invoke({"input": oo_domain_query})

print("--- QUERY ---")
print(oo_domain_query)
print("\n--- BASELINE ANSWER (temperature=0, fallback rule ON) ---")
print(baseline_response["answer"])
print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(baseline_response["context"]):
    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))
    print("   Preview:", doc.page_content[:120].replace("\n", " "), "...")

--- QUERY ---
What is the average surface temperature of the planet Venus?

--- BASELINE ANSWER (temperature=0, fallback rule ON) ---
I cannot answer based on the provided domain data.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/CHAPTER II.txt
   Preview: The presence of at least a majority of the members shall constitute a quorum. The  chairman, or in his absence the vice- ...
Chunk 2 Source: my_data/CHAPTER VII.txt
   Preview: 1595 was finally passed by the House of Representatives and the Senate on December  20, 2000 and December 12, 2000, resp ...
Chunk 3 Source: my_data/CHAPTER VI.txt
   Preview: pesos (P100,000.00) but not more than One million pesos (P1,000,000.00),  or imprisonment not less than one (1) year but ...


### Task 2 result

**Query used:**

> What is the average surface temperature of the planet Venus?

**Actual output:**

> `I cannot answer based on the provided domain data.`

**Did the system return the exact fallback message?** **Yes** — verbatim, character for character.

**The more informative half of this output is the retrieved chunks.** The three chunks returned were:

| # | Source | Content |
|---|--------|---------|
| 1 | `CHAPTER II.txt` | Commission quorum and chairmanship rules |
| 2 | `CHAPTER VII.txt` | Dates the bill was passed by the House and Senate |
| 3 | `CHAPTER VI.txt` | Penalty amounts for various prohibited acts |

None of these has the slightest connection to planetary science. The retriever returned them anyway, because **the retrieval step cannot refuse.** It performs nearest-neighbour search and returns the top `k` vectors regardless of how distant those neighbours are; there is no similarity threshold below which it declines to answer, and no concept of "nothing here is relevant."

This is the single most important thing Task 2 demonstrates, and it sets up Task 3 exactly. The refusal behaviour is **entirely a property of the prompt**, not of the retriever or the vector store. Every safeguard visible in this output lives in one instruction written in plain English in Cell 5 — which is precisely the safeguard Task 3 goes on to remove.

---
## Task 3: Hallucination Stress-Test

**Method.** Two things change at once relative to the baseline:

1. `temperature` goes from `0` to `1.0` — sampling is no longer greedy, so lower-probability continuations become reachable.
2. The fallback instruction is deleted from `system_prompt` — the model loses its sanctioned way to decline.

The same out-of-domain question from Task 2 is then re-asked.

> **Note on experimental design:** because two variables change together, a difference in output cannot be attributed to either one alone. This is a limitation of the prescribed procedure and is worth one sentence in the report's analysis. If you have time, running the two changes separately (temperature only, then prompt only) makes for a much stronger Section III — the prompt change is usually the one that does the work.

**Re-running is mandatory.** `rag_chain` captured the old prompt and LLM when Cell 6 ran. The three cells below re-execute Cells 5, 6 and 7 with the modified configuration into a **separate** variable (`rag_chain_ungrounded`), so your baseline chain stays intact and both configurations remain callable for a fair side-by-side comparison.

### Cell 5b — Modified model and prompt (temperature=1.0, fallback rule deleted)

Two deliberate changes from Cell 5, both required by the task:

- `temperature=1.0` instead of `temperature=0`.
- The line beginning `"If the answer cannot be found in the context, reply: ..."` is **removed**.

Everything else — model name, role line, restriction line, `{context}` placeholder, message structure — is identical, so the comparison isolates the two intended changes.

In [ ]:
# Cell 5b: Model Initialization and Domain System Prompt (UNGROUNDED CONFIGURATION)
# CHANGE 1: temperature 0 -> 1.0
llm_ungrounded = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=1.0
)

# CHANGE 2: the fallback instruction line has been deleted
system_prompt_ungrounded = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n\n"
    "Context:\n{context}"
)

prompt_ungrounded = ChatPromptTemplate.from_messages([
    ("system", system_prompt_ungrounded),
    ("human", "{input}"),
])

### Cell 6b — Re-assemble the pipeline

`create_stuff_documents_chain` and `create_retrieval_chain` are called again so the new prompt and new LLM are actually bound into a chain. The **same** `retriever` object is reused deliberately: retrieval is held constant, so any behavioural difference comes from the generation half alone.

In [ ]:
# Cell 6b: Pipeline Assembly (UNGROUNDED CONFIGURATION)
combine_docs_chain_ungrounded = create_stuff_documents_chain(llm_ungrounded, prompt_ungrounded)

# Same retriever, new prompt + new LLM
rag_chain_ungrounded = create_retrieval_chain(retriever, combine_docs_chain_ungrounded)

### Cell 7b — Re-run the identical out-of-domain query

The cell below runs **both** chains on the same question so the two outputs appear side by side in one output block — convenient for pasting into Table I of the report.

Because `temperature=1.0` is stochastic, the ungrounded configuration is invoked three times. A single sample tells you very little about a sampled distribution: the run-to-run variation is itself part of the finding, and reporting three trials is far more defensible than reporting one.

In [ ]:
# Cell 7b: Hallucination Stress-Test — same query, both configurations
print("=" * 70)
print("QUERY:", oo_domain_query)
print("=" * 70)

print("\n--- CONFIG A: BASELINE (temperature=0, fallback rule ON) ---")
print(baseline_response["answer"])

print("\n--- CONFIG B: UNGROUNDED (temperature=1.0, fallback rule DELETED) ---")
for trial in range(3):
    ungrounded_response = rag_chain_ungrounded.invoke({"input": oo_domain_query})
    print(f"\n[Trial {trial + 1}]")
    print(ungrounded_response["answer"])

print("\n" + "=" * 70)
print("Retrieved chunks are IDENTICAL for both configurations (same retriever, same query).")

QUERY: What is the average surface temperature of the planet Venus?

--- CONFIG A: BASELINE (temperature=0, fallback rule ON) ---
I cannot answer based on the provided domain data.

--- CONFIG B: UNGROUNDED (temperature=1.0, fallback rule DELETED) ---

[Trial 1]
I’m sorry, but the information you’re looking for isn’t included in the context that was provided.

[Trial 2]
I’m sorry, but I don’t have that information in the provided context.

[Trial 3]
I’m sorry, but the information you’re looking for isn’t included in the context that was provided.

Retrieved chunks are IDENTICAL for both configurations (same retriever, same query).


---
### Additional configuration: Config C — restriction line also removed
### (additional analysis)

The Task 3 procedure changes two things simultaneously: temperature goes from 0 to 1.0, *and* the fallback sentence is deleted. Because both move together, the result cannot show which one is responsible. Config B above refused all three times, which already suggests temperature was not the decisive factor — but it cannot prove it.

Config C isolates the variable. Temperature stays at 1.0, exactly as in Config B, and the remaining restriction — `"Answer questions strictly using ONLY the provided context below"` — is deleted as well, leaving only a generic assistant instruction and the context block.

If Config C behaves like Config B, the restriction line was not what held the model in place. If it answers the Venus question, the restriction line was doing all the work and temperature was never the mechanism.

In [ ]:
# Config C: restriction line ALSO removed
system_prompt_free = "You are a helpful assistant.\n\nContext:\n{context}"
prompt_free = ChatPromptTemplate.from_messages([
    ("system", system_prompt_free),
    ("human", "{input}"),
])
chain_free = create_retrieval_chain(retriever, create_stuff_documents_chain(llm_ungrounded, prompt_free))

print("--- CONFIG C: no restriction, no fallback, temp=1.0 ---")
for t in range(3):
    print(f"\n[Trial {t+1}]")
    print(chain_free.invoke({"input": oo_domain_query})["answer"])

--- CONFIG C: no restriction, no fallback, temp=1.0 ---

[Trial 1]
The surface of Venus is extremely hot—its average temperature is about **735 K** (roughly **462 °C** or **870 °F**). This makes Venus the hottest planet in our solar system, largely because of its dense CO₂ atmosphere and runaway greenhouse effect.

[Trial 2]
Venus’s surface is blisteringly hot—its average temperature is about **460 – 470 °C** (≈ **737 – 743 K**), which is roughly **860 – 870 °F**. This extreme heat is the result of a runaway greenhouse effect that traps almost all of the star’s sunlight in a dense CO₂‑rich atmosphere.

[Trial 3]
The average surface temperature of Venus is about **464 °C (≈737 K)**.


### Step 3 — Observe & Compare

**Table I — Test Queries and Model Outputs**

| # | Query | Type | Expected | Config A (temp=0, fallback ON) | Config B (temp=1.0, fallback OFF) | Config C (temp=1.0, no restriction) |
|---|-------|------|----------|-------------------------------|-----------------------------------|--------------------------------------|
| 1 | Members of the National Solid Waste Management Commission | In-domain | 14 government, 3 private sector | Correct; sourced to `CHAPTER II.txt` | — | — |
| 2 | Sources of the Solid Waste Management Fund | In-domain | Fines, permits, donations, GAA appropriations | Correct; sourced to `CHAPTER V.txt` | — | — |
| 3 | Penalty for littering in a public place | In-domain | ₱300–₱1,000 or 1–15 days community service | **False refusal** (see investigation) | — | — |
| 4 | Average surface temperature of Venus | Out-of-domain | Fallback message | Exact fallback message | Refused, varied wording | **Answered in full: ~735 K / 462 °C** |

**Configuration summary**

| Config | Restriction line | Fallback line | Temperature | Outcome |
|--------|------------------|---------------|-------------|---------|
| A | present | present | 0 | Exact scripted fallback |
| B | present | deleted | 1.0 | Refused all 3 trials, in its own words |
| C | **deleted** | deleted | 1.0 | Answered from training data, all 3 trials |

---

#### **1. Did the model remain grounded in the retrieved context?**

Under Configs A and B, yes — completely. Under Config C, no. The Config C answers concern Venus's atmosphere and greenhouse effect, and not one word of them is traceable to the three chunks of Philippine waste-management law sitting in the context window. Those chunks were supplied and silently ignored.

#### **2. Did it introduce unsupported information?**

Under Config C, everything it produced was unsupported by the context. The specific claims — an average surface temperature of about 735 K, a dense CO₂ atmosphere, a runaway greenhouse effect, Venus being the hottest planet in the solar system — came entirely from the model's training data.

These answers are factually correct, and that is the point worth dwelling on. Venus really does average roughly 464 °C. So this is not hallucination in the sense of fabrication; it is ungroundedness, which is the more insidious failure. The system stopped being a document-based assistant and became a general chatbot, and nothing in the output marks the transition. A user has no way to tell that this answer did not come from the corpus. On a question where the model's training data happened to be wrong, stale, or contested, the identical mechanism would produce a confident falsehood in exactly the same presentation — and the retrieved sources printed alongside it would lend it entirely unearned credibility.

#### **3. Did it refuse anyway?**

Under Config B, yes — all three trials. This is a genuine result, not a failed experiment, and forcing it into a "the model started hallucinating" narrative would misreport the data.

The explanation is straightforward. Task 3 instructs the deletion of the fallback *sentence* only. The line `Answer questions strictly using ONLY the provided context below` remains in `system_prompt_ungrounded`. The model therefore still knew it was confined to the context; what it lost was the scripted wording for declining, not permission to decline. That is precisely what the output shows — it refused in its own phrasing rather than the prescribed sentence.

Config C confirms this. Deleting the restriction line as well, with temperature held constant at 1.0, changed the behaviour completely.

#### **4. Consistency across trials**

Config A is deterministic: the same input produces the same string every time.

Config B produced two distinct phrasings across three runs — trials 1 and 3 matched, trial 2 differed. Same input, same context, different output: sampling variance made visible only because the cell ran three trials rather than one.

Config C varied on **substance**, not just phrasing: 735 K in trial 1, a hedged range of 460–470 °C in trial 2, and 464 °C in trial 3. The same question produced three different numbers. This is the clearest demonstration in the whole experiment of what temperature does once the model is drawing on parametric memory instead of a fixed context — the variation moves from wording into facts.

#### **5. Which change mattered more?**

The prompt restriction. The evidence is the controlled comparison between B and C: temperature is 1.0 in both, and the only difference is the presence of the restriction line. B refused; C answered. Temperature therefore cannot be the mechanism that keeps the model grounded.

The conclusion the data supports is this: temperature governs how the model says things; the prompt governs what it is permitted to say. Raising temperature alone did not produce hallucination because the substantive constraint was left intact. It changed the wording of a refusal, not the decision to refuse.

#### **Limitation of the prescribed procedure**

The two-configuration comparison the lab specifies changes two independent variables simultaneously, so on its own it cannot attribute any observed difference to either one. Config C was added specifically to break that confound, and it is what makes the conclusion above defensible rather than merely plausible.

#### **A note on the false refusal**

Query 3 shows the cost side of the ledger. Strict grounding does not only prevent wrong answers — it also blocks correct ones when retrieval fails to surface the supporting text, and the user cannot distinguish the two cases from the output. Both produce the identical fallback message. A system tuned only to avoid hallucination will refuse questions it could have answered, and that is a real usability cost, not a free win.